In [45]:
from utils import * 
import sympy as sp
import numpy as np
from itertools import product

In [100]:
def create_variables(edge_list):
    return {(u, v): sp.Symbol(f"m{u}{v}") for u, v in edge_list}


def get_M_matrix(edge_list, n):
    vars = create_variables(edge_list)
    M = sp.zeros(n)

    for (u, v), var in vars.items():
        M[u-1, v-1] = var

    return M, list(vars.values())

def get_L_matrix(edge_list, n, low=-1.0, high=1.0):
    L = np.zeros((n, n))

    for u, v in edge_list:
        L[u-1, v-1] = np.random.uniform(low, high)

    return L

def get_J(B2, n):
    B2 = set(B2)
    return [(i, j) for i in range(1, n+1)
                   for j in range(i+1, n+1)
                   if (i, j) not in B2]

def get_K(B1, n):
    symmetric_B1 = set(B1) | {(j, i) for i, j in B1}
    return [(i, j)
            for i in range(1, n+1)
            for j in range(1, n+1)
            if i == j or (i, j) in symmetric_B1]

def map_pairs(JK):
    return [((a, c), (b, d)) for ((a, b), (c, d)) in JK]

def filter_pairs(JK, A):
    return [((a, b), (c, d)) 
            for ((a, b), (c, d)) in JK
            if A[a-1, b-1] != 0 and A[c-1, d-1] != 0]

def create_equations(JK, A):
    return [
        A[a-1, b-1] * A[c-1, d-1]
        for ((a, b), (c, d)) in JK
    ]

In [ ]:
# Number of Vertices
n = 3

# Generate DAGs
G1 = nx.DiGraph()
G2 = nx.DiGraph()
V = range(1,n+1)
G1.add_nodes_from(V)
G2.add_nodes_from(V)

# Edge Sets
E1 = [(1,2),(2,3)]
E2 = [(1,2),(1,3)]
B1 = [(2,3)]
B2 = [(2,3)]

# Add Edges
G1.add_edges_from(E1)
G2.add_edges_from(E2)

In [101]:
L = get_L_matrix(E1, n)
M, vars = get_M_matrix(E2, n)
L, M, vars

(array([[ 0.        ,  0.9590586 ,  0.        ],
        [ 0.        ,  0.        , -0.56074488],
        [ 0.        ,  0.        ,  0.        ]]),
 Matrix([
 [0, m12, m13],
 [0,   0,   0],
 [0,   0,   0]]),
 [m12, m13])

In [33]:
I = np.eye(n)
A = (I-M).T @ np.linalg.inv(I - L).T
A

Matrix([
[                         1.0,                  0,   0],
[-1.0*m12 - 0.975296378495085,                1.0,   0],
[ 0.214161468033346 - 1.0*m13, -0.219586038414092, 1.0]])

In [46]:
J = get_J(B2,n)
K = get_K(B1,n)
J, K 

([(1, 2), (1, 3)], [(1, 1), (2, 2), (2, 3), (3, 2), (3, 3)])

In [71]:
JK = map_pairs(list(product(J,K)))
JK

[((1, 1), (2, 1)),
 ((1, 2), (2, 2)),
 ((1, 2), (2, 3)),
 ((1, 3), (2, 2)),
 ((1, 3), (2, 3)),
 ((1, 1), (3, 1)),
 ((1, 2), (3, 2)),
 ((1, 2), (3, 3)),
 ((1, 3), (3, 2)),
 ((1, 3), (3, 3))]

In [75]:
JK_reduced = filter_pairs(JK, A)
JK_reduced

[((1, 1), (2, 1)), ((1, 1), (3, 1))]

In [84]:
eq = create_equations(JK_reduced, A)
eq

[-1.0*m12 - 0.975296378495085, 0.214161468033346 - 1.0*m13]

In [ ]:
vars = list(vars.values())

[m12, m13]

In [104]:
sp.solve(eq, vars)

{m12: -0.975296378495085, m13: 0.214161468033346}

In [217]:
num_vertex = 3
num_confounding = 3
n = num_vertex

G1 = random_dag(num_vertex,random.random())
B1 = random_confounding(num_vertex,num_confounding)
G2 = random_dag(num_vertex,random.random())
B2 = random_confounding(num_vertex,num_confounding)

L = get_L_matrix(list(G1.edges()), n)
M, vars = get_M_matrix(list(G2.edges()), n)
L, M, vars

I = np.eye(n)
A = (I-M).T @ np.linalg.inv(I - L).T
J = get_J(B2,n)
K = get_K(B1,n)
JK = map_pairs(list(product(J,K)))
JK_reduced = filter_pairs(JK, A)
eq = create_equations(JK_reduced, A)
sp.solve(eq, vars)

[]

In [213]:
B2

[]

In [220]:
eq

[]

In [218]:
check_inclusion_full(G1,G2,B1,B2)

True